# YOLOX Polygon Training: Thermal Cheetah Dataset

This notebook demonstrates how to train YOLOX with **polygon bounding box** support using the Thermal Cheetah dataset.

### Prerequisites
- Ensure you have cloned the repository with the dataset (already included in this branch).
- **Local PC**: Select the `.venv` kernel (top right in VS Code or Jupyter).
- **Google Colab**: Ensure GPU is enabled (though the code now supports CPU for testing).

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path to find 'yolox' module
project_root = str(Path(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Current working directory: {os.getcwd()}")
print(f"Python Path updated with: {project_root}")

try:
    import yolox
    import loguru
    print("YOLOX and dependencies found successfully!")
except ImportError:
    print("Dependencies missing. Installing...")
    !pip install -r requirements.txt
    !pip install -e .
    import yolox
    print("YOLOX module installed and imported.")

## 1. Setup and Verification

First, let's verify the polygon annotations by drawing them on some sample images.

In [ ]:
# Run visualization tool to verify polygon annotations
!python tools/visualize_polygons.py --json "datasets/Thermal Cheetah.v1-square.coco/train/instances_train2017_poly.json" --img_dir "datasets/Thermal Cheetah.v1-square.coco/train" --out_dir "vis_outputs" --num 3

## 2. Polygon Functionality Check

Run the verification script to ensure polygon IoU, NMS, and loss functions are working correctly.

In [ ]:
!python tools/verify_polygon.py

## 3. Training with Polygon Support

We use the pre-configured experiment file `exps/example/yolox_thermal_cheetah_poly.py` which has `self.use_polygon = True` enabled.

In [ ]:
# Start training
# Note: On local CPU, we use -b 2 and no GPU devices flag
import torch
device_flag = "-d 1" if torch.cuda.is_available() else ""
amp_flag = "--fp16" if torch.cuda.is_available() else ""

!python tools/train.py -f exps/example/yolox_thermal_cheetah_poly.py {device_flag} -b 2 {amp_flag} -o -c yolox_s.pth --experiment-name output_local

## 4. Evaluation

After training, you can evaluate the model on the test set.

In [ ]:
# Evaluate on test set
!python tools/eval.py -f exps/example/yolox_thermal_cheetah_poly.py -c YOLOX_outputs/output_local/best_ckpt.pth -b 8 -d 1 --conf 0.001

## 5. Inference & Visualization

Finally, run inference on a test image to see the polygon predictions.

In [ ]:
# Run demo inference
!python tools/demo.py image -f exps/example/yolox_thermal_cheetah_poly.py \
    -c YOLOX_outputs/output_local/best_ckpt.pth \
    --path datasets/Thermal Cheetah.v1-square.coco/test/ \
    --conf 0.3 --nms 0.45 --tsize 640 --save_result --device cpu